In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# ============================================================
# NB_Silver_Data_Quality
# Microsoft Fabric - Insurance Data Quality
#
# Lakehouse : LH_Silver
#
# Purpose:
#   1. Read Silver Delta tables
#   2. Apply business/data-quality rules
#   3. Identify invalid records
#   4. Create reject/quarantine tables
#   5. Create DQ audit results
#   6. Validate results
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    when,
    concat_ws
)
from datetime import datetime
import uuid


# ============================================================
# 1. CONFIGURATION
# ============================================================

SILVER_LAKEHOUSE = "LH_Silver"
SCHEMA = "dbo"

RUN_ID = str(uuid.uuid4())

print("================================================")
print("SILVER DATA QUALITY PROCESS")
print("================================================")
print(f"Run ID: {RUN_ID}")


# ============================================================
# 2. READ SILVER TABLES
# ============================================================

customers = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_customers"
)

policies = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_policies"
)

claims = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_claims"
)

payments = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_payments"
)

print("\nSilver tables loaded successfully.")

print(f"Customers : {customers.count()}")
print(f"Policies  : {policies.count()}")
print(f"Claims    : {claims.count()}")
print(f"Payments  : {payments.count()}")


# ============================================================
# 3. DISPLAY ACTUAL SCHEMAS
# ============================================================

print("\nCUSTOMERS COLUMNS")
print(customers.columns)

print("\nPOLICIES COLUMNS")
print(policies.columns)

print("\nCLAIMS COLUMNS")
print(claims.columns)

print("\nPAYMENTS COLUMNS")
print(payments.columns)


# ============================================================
# 4. CUSTOMER DATA QUALITY
# ============================================================

print("\nChecking Customers...")

customers_dq = (
    customers

    .withColumn(
        "_dq_error",
        concat_ws(
            "; ",

            when(
                col("customer_id").isNull(),
                lit("CUSTOMER_ID_NULL")
            ),

            when(
                col("email").isNull(),
                lit("EMAIL_NULL")
            ),

            when(
                col("email").isNotNull() &
                (~col("email").contains("@")),
                lit("INVALID_EMAIL")
            ),

            when(
                col("state").isNull(),
                lit("STATE_NULL")
            )
        )
    )
)


customer_rejects = (
    customers_dq
    .filter(
        col("_dq_error") != ""
    )
    .withColumn("_dq_run_id", lit(RUN_ID))
    .withColumn("_dq_processed_ts", current_timestamp())
)


# ============================================================
# 5. POLICY DATA QUALITY
# ============================================================

print("Checking Policies...")

# Valid customer keys
customer_keys = (
    customers
    .select("customer_id")
    .dropDuplicates()
    .withColumn("_customer_exists", lit(1))
)


policies_dq = (
    policies

    .join(
        customer_keys,
        "customer_id",
        "left"
    )

    .withColumn(
        "_dq_error",
        concat_ws(
            "; ",

            when(
                col("policy_id").isNull(),
                lit("POLICY_ID_NULL")
            ),

            when(
                col("customer_id").isNull(),
                lit("CUSTOMER_ID_NULL")
            ),

            when(
                col("_customer_exists").isNull(),
                lit("CUSTOMER_NOT_FOUND")
            ),

            when(
                col("annual_premium") < 0,
                lit("NEGATIVE_ANNUAL_PREMIUM")
            ),

            when(
                col("coverage_limit") < 0,
                lit("NEGATIVE_COVERAGE_LIMIT")
            ),

            when(
                col("deductible") < 0,
                lit("NEGATIVE_DEDUCTIBLE")
            ),

            when(
                col("policy_end_date") <
                col("policy_start_date"),
                lit("INVALID_POLICY_DATE_RANGE")
            )
        )
    )
)


policy_rejects = (
    policies_dq

    .filter(
        col("_dq_error") != ""
    )

    .drop("_customer_exists")

    .withColumn(
        "_dq_run_id",
        lit(RUN_ID)
    )

    .withColumn(
        "_dq_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 6. CLAIM DATA QUALITY
# ============================================================

print("Checking Claims...")


policy_keys = (
    policies
    .select("policy_id")
    .dropDuplicates()
    .withColumn("_policy_exists", lit(1))
)


claim_customer_keys = (
    customers
    .select("customer_id")
    .dropDuplicates()
    .withColumn("_claim_customer_exists", lit(1))
)


claims_dq = (
    claims

    .join(
        policy_keys,
        "policy_id",
        "left"
    )

    .join(
        claim_customer_keys,
        "customer_id",
        "left"
    )

    .withColumn(
        "_dq_error",
        concat_ws(
            "; ",

            when(
                col("claim_id").isNull(),
                lit("CLAIM_ID_NULL")
            ),

            when(
                col("policy_id").isNull(),
                lit("POLICY_ID_NULL")
            ),

            when(
                col("_policy_exists").isNull(),
                lit("POLICY_NOT_FOUND")
            ),

            when(
                col("customer_id").isNull(),
                lit("CUSTOMER_ID_NULL")
            ),

            when(
                col("_claim_customer_exists").isNull(),
                lit("CUSTOMER_NOT_FOUND")
            ),

            when(
                col("claim_amount") < 0,
                lit("NEGATIVE_CLAIM_AMOUNT")
            ),

            when(
                col("approved_amount") < 0,
                lit("NEGATIVE_APPROVED_AMOUNT")
            ),

            when(
                col("approved_amount") >
                col("claim_amount"),
                lit("APPROVED_EXCEEDS_CLAIM")
            ),

            when(
                col("incident_date") >
                col("claim_date"),
                lit("INCIDENT_AFTER_CLAIM_DATE")
            )
        )
    )
)


claim_rejects = (
    claims_dq

    .filter(
        col("_dq_error") != ""
    )

    .drop(
        "_policy_exists",
        "_claim_customer_exists"
    )

    .withColumn(
        "_dq_run_id",
        lit(RUN_ID)
    )

    .withColumn(
        "_dq_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 7. PAYMENT DATA QUALITY
# ============================================================

print("Checking Payments...")


# We first build rules that are safe for the known
# payment fields from the Silver transformation.

payments_dq = (
    payments

    .withColumn(
        "_dq_error",
        concat_ws(
            "; ",

            when(
                col("payment_id").isNull(),
                lit("PAYMENT_ID_NULL")
            ),

            when(
                col("payment_amount").isNull(),
                lit("PAYMENT_AMOUNT_NULL")
            ),

            when(
                col("payment_amount") < 0,
                lit("NEGATIVE_PAYMENT_AMOUNT")
            ),

            when(
                col("payment_date").isNull(),
                lit("PAYMENT_DATE_NULL")
            )
        )
    )
)


payment_rejects = (
    payments_dq

    .filter(
        col("_dq_error") != ""
    )

    .withColumn(
        "_dq_run_id",
        lit(RUN_ID)
    )

    .withColumn(
        "_dq_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 8. CALCULATE DATA QUALITY METRICS
# ============================================================

print("\nCalculating DQ metrics...")


customer_total = customers.count()
policy_total = policies.count()
claim_total = claims.count()
payment_total = payments.count()

customer_failed = customer_rejects.count()
policy_failed = policy_rejects.count()
claim_failed = claim_rejects.count()
payment_failed = payment_rejects.count()


def failure_pct(failed, total):
    if total == 0:
        return 0.0

    return round(
        (failed / total) * 100,
        2
    )


dq_metrics = [

    (
        RUN_ID,
        "silver_customers",
        customer_total,
        customer_failed,
        customer_total - customer_failed,
        failure_pct(
            customer_failed,
            customer_total
        )
    ),

    (
        RUN_ID,
        "silver_policies",
        policy_total,
        policy_failed,
        policy_total - policy_failed,
        failure_pct(
            policy_failed,
            policy_total
        )
    ),

    (
        RUN_ID,
        "silver_claims",
        claim_total,
        claim_failed,
        claim_total - claim_failed,
        failure_pct(
            claim_failed,
            claim_total
        )
    ),

    (
        RUN_ID,
        "silver_payments",
        payment_total,
        payment_failed,
        payment_total - payment_failed,
        failure_pct(
            payment_failed,
            payment_total
        )
    )
]


dq_results = spark.createDataFrame(
    dq_metrics,
    [
        "run_id",
        "table_name",
        "records_checked",
        "records_failed",
        "records_passed",
        "failure_percentage"
    ]
).withColumn(
    "processed_ts",
    current_timestamp()
)


# ============================================================
# 9. CREATE / REPLACE REJECT TABLES
# ============================================================

print("\nWriting reject tables...")


def write_delta_table(df, table_name):

    full_name = (
        f"{SILVER_LAKEHOUSE}."
        f"{SCHEMA}."
        f"{table_name}"
    )

    print(
        f"Creating/Replacing: {full_name}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(full_name)
    )

    print(
        f"SUCCESS: {full_name}"
    )


write_delta_table(
    customer_rejects,
    "reject_customers"
)

write_delta_table(
    policy_rejects,
    "reject_policies"
)

write_delta_table(
    claim_rejects,
    "reject_claims"
)

write_delta_table(
    payment_rejects,
    "reject_payments"
)


# ============================================================
# 10. WRITE DQ RESULTS
# ============================================================

print("\nWriting DQ results...")


(
    dq_results.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{SILVER_LAKEHOUSE}.{SCHEMA}.dq_results"
    )
)


# ============================================================
# 11. DISPLAY DATA QUALITY SUMMARY
# ============================================================

print("\n==============================================")
print("DATA QUALITY SUMMARY")
print("==============================================")

display(
    dq_results.orderBy(
        "table_name"
    )
)


# ============================================================
# 12. DISPLAY REJECT COUNTS
# ============================================================

print("\nReject counts:")

print(
    f"Customers : {customer_failed}"
)

print(
    f"Policies  : {policy_failed}"
)

print(
    f"Claims    : {claim_failed}"
)

print(
    f"Payments  : {payment_failed}"
)


# ============================================================
# 13. SAMPLE REJECTED RECORDS
# ============================================================

print("\nSample rejected claims:")

display(
    claim_rejects
    .select(
        "claim_id",
        "policy_id",
        "customer_id",
        "_dq_error"
    )
    .limit(20)
)


# ============================================================
# 14. COMPLETION
# ============================================================

print("\n==============================================")
print("SILVER DATA QUALITY PROCESS COMPLETED")
print("==============================================")

print("Created/Replaced:")
print("LH_Silver.dbo.reject_customers")
print("LH_Silver.dbo.reject_policies")
print("LH_Silver.dbo.reject_claims")
print("LH_Silver.dbo.reject_payments")

print("\nAudit table:")
print("LH_Silver.dbo.dq_results")

print("==============================================")

StatementMeta(, 7aded380-a270-468e-9e03-099371e5e397, 3, Finished, Available, Finished, False)

SILVER DATA QUALITY PROCESS
Run ID: bb094abf-380f-40fb-ab26-30c206bf54ee

Silver tables loaded successfully.
Customers : 500
Policies  : 750
Claims    : 1200
Payments  : 1500

CUSTOMERS COLUMNS
['customer_id', 'first_name', 'last_name', 'date_of_birth', 'email', 'phone', 'address', 'city', 'state', 'zip_code', 'created_date', 'customer_status', '_silver_processed_ts']

POLICIES COLUMNS
['policy_id', 'customer_id', 'product_type', 'policy_start_date', 'policy_end_date', 'annual_premium', 'coverage_limit', 'deductible', 'policy_status', 'agent_id', 'last_updated', '_silver_processed_ts']

CLAIMS COLUMNS
['claim_id', 'policy_id', 'customer_id', 'claim_date', 'incident_date', 'claim_type', 'claim_amount', 'approved_amount', 'claim_status', 'description', 'reported_channel', 'adjuster_id', 'last_updated', '_silver_processed_ts']

PAYMENTS COLUMNS
['payment_id', 'policy_id', 'customer_id', 'claim_id', 'payment_date', 'payment_type', 'payment_amount', 'payment_method', 'payment_status', 'tran

SynapseWidget(Synapse.DataFrame, 44cc7d88-7e1c-4c85-8b12-73e48f0c03b3)


Reject counts:
Customers : 1
Policies  : 1
Claims    : 2
Payments  : 1

Sample rejected claims:


SynapseWidget(Synapse.DataFrame, 0952fc72-fd44-46d5-bd17-c75d39277f08)


SILVER DATA QUALITY PROCESS COMPLETED
Created/Replaced:
LH_Silver.dbo.reject_customers
LH_Silver.dbo.reject_policies
LH_Silver.dbo.reject_claims
LH_Silver.dbo.reject_payments

Audit table:
LH_Silver.dbo.dq_results
